# DAPO Training

Use this notebook to run the first offline DAPO-style RLVR training pass from student rollouts. It restores the selected SFT checkpoint from Google Drive, loads rollout data from Drive, trains `scripts/train_dapo.py`, and saves the DAPO adapter back to Drive.

## 1. Pull Repo / Setup

In [ ]:
import importlib.util
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/kdnehihi/strategy-distill-rl.git"
REPO_DIR = Path("/content/strategy-distill-rl")
IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    if REPO_DIR.exists():
        print(f"Pulling latest repo in {REPO_DIR}")
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", "main"], check=True)
    else:
        print(f"Cloning repo to {REPO_DIR}")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
else:
    print(f"Local run detected. Current directory: {Path.cwd()}")

PROJECT_ROOT = Path.cwd()
print("Working directory:", PROJECT_ROOT)

## 2. Install Dependencies

In [ ]:
AUTO_INSTALL_MISSING_DEPENDENCIES = True

REQUIRED_PACKAGES = {
    "peft": "peft",
    "accelerate": "accelerate",
    "transformers": "transformers",
    "tqdm": "tqdm",
    "pandas": "pandas",
}


def install_missing_dependencies():
    import importlib.util
    import sys

    missing = [
        package_name
        for import_name, package_name in REQUIRED_PACKAGES.items()
        if importlib.util.find_spec(import_name) is None
    ]
    if not missing:
        print("All DAPO dependencies are already installed.")
        return

    if not AUTO_INSTALL_MISSING_DEPENDENCIES:
        raise ModuleNotFoundError("Missing packages: " + ", ".join(missing))

    import sys
    print("Installing missing dependencies:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)


def parse_version_tuple(version):
    parts = []
    for chunk in version.split(".")[:3]:
        try:
            parts.append(int(chunk))
        except ValueError:
            parts.append(0)
    while len(parts) < 3:
        parts.append(0)
    return tuple(parts)


def fix_incompatible_torchao():
    import importlib.metadata
    import sys

    try:
        version = importlib.metadata.version("torchao")
    except importlib.metadata.PackageNotFoundError:
        print("torchao is not installed; no compatibility fix needed.")
        return

    if parse_version_tuple(version) >= (0, 16, 0):
        print(f"torchao {version} is compatible.")
        return

    print(f"Found torchao {version}; uninstalling because PEFT LoRA does not need it here.")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)


install_missing_dependencies()
fix_incompatible_torchao()

## 3. Config

In [ ]:
from pathlib import Path

MODEL_NAME = "Qwen/Qwen2.5-Math-1.5B-Instruct"

# Drive locations used in the previous notebooks.
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/RL/Data")
DRIVE_CHECKPOINT_DIR = Path("/content/drive/MyDrive/RL/Checkpoints")

# This can be either the zip you saved earlier or an extracted adapter folder.
DRIVE_SFT_CHECKPOINT_SOURCE = DRIVE_CHECKPOINT_DIR / "balanced_r16_a32_4000_20260630_233638.zip"
SFT_CHECKPOINT_NAME = "balanced_r16_a32_4000"
LOCAL_SFT_ADAPTER_PATH = Path("checkpoints/student_sft") / SFT_CHECKPOINT_NAME

# Use the full rollout file for DAPO so wrong/invalid outputs can still provide contrast.
ROLLOUT_FILE_NAME = "rl_rollouts_student.jsonl"
ROLLOUT_PATH = Path("data") / ROLLOUT_FILE_NAME

RUN_DAPO_TRAINING = True
USE_REFERENCE_MODEL = True

# Start small. Increase after this notebook runs cleanly.
MAX_GROUPS = 200
BATCH_SIZE = 1
EPOCHS = 1
LEARNING_RATE = 1e-6
MAX_LENGTH = 1024
CLIP_LOW = 0.2
CLIP_HIGH = 0.28

DAPO_RUN_NAME = "dapo_debug_200"
DAPO_OUTPUT_DIR = Path("checkpoints/student_dapo") / DAPO_RUN_NAME
SAVE_DAPO_TO_DRIVE = True

print("SFT checkpoint source:", DRIVE_SFT_CHECKPOINT_SOURCE)
print("Rollout source:", DRIVE_DATA_DIR / ROLLOUT_FILE_NAME)
print("DAPO output:", DAPO_OUTPUT_DIR)

## 4. Helpers

In [ ]:
import json
import shutil
import subprocess
import zipfile
from datetime import datetime
from pathlib import Path

import pandas as pd


def run_command(args):
    command = [str(arg) for arg in args]
    print("$", " ".join(command))
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    output_lines = []
    for line in process.stdout:
        print(line, end="")
        output_lines.append(line)
    return_code = process.wait()
    if return_code != 0:
        tail = "".join(output_lines[-80:])
        raise RuntimeError(
            f"Command failed with exit code {return_code}: {' '.join(command)}\n"
            f"Last output lines:\n{tail}"
        )


def mount_drive_if_needed():
    if not IN_COLAB:
        return
    if Path("/content/drive/MyDrive").exists():
        print("Google Drive already mounted.")
        return
    from google.colab import drive
    drive.mount("/content/drive")


def read_jsonl(path, limit=None):
    rows = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            if limit is not None and len(rows) >= limit:
                break
            rows.append(json.loads(line))
    return rows


def find_adapter_source_dir(root: Path) -> Path:
    candidates = [root]
    candidates.extend(path for path in root.rglob("*") if path.is_dir())
    for candidate in candidates:
        if (candidate / "adapter_config.json").exists():
            return candidate
    raise FileNotFoundError(f"Could not find adapter_config.json under {root}")


def restore_sft_checkpoint():
    mount_drive_if_needed()
    source = Path(DRIVE_SFT_CHECKPOINT_SOURCE)
    target = Path(LOCAL_SFT_ADAPTER_PATH)

    if (target / "adapter_config.json").exists():
        print(f"SFT adapter already restored: {target}")
        return target

    if not source.exists():
        raise FileNotFoundError(f"Missing SFT checkpoint source: {source}")

    target.parent.mkdir(parents=True, exist_ok=True)
    if source.is_dir():
        adapter_dir = find_adapter_source_dir(source)
        shutil.copytree(adapter_dir, target, dirs_exist_ok=True)
        print(f"Copied SFT adapter folder: {adapter_dir} -> {target}")
        return target

    extract_dir = Path("/tmp/strategy_distill_sft_checkpoint") / SFT_CHECKPOINT_NAME
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    print(f"Extracting SFT checkpoint archive: {source}")
    with zipfile.ZipFile(source, "r") as zf:
        zf.extractall(extract_dir)

    adapter_dir = find_adapter_source_dir(extract_dir)
    shutil.copytree(adapter_dir, target, dirs_exist_ok=True)
    print(f"Restored SFT adapter: {adapter_dir} -> {target}")
    return target


def copy_rollout_data():
    mount_drive_if_needed()
    Path("data").mkdir(parents=True, exist_ok=True)
    source = Path(DRIVE_DATA_DIR) / ROLLOUT_FILE_NAME
    target = Path(ROLLOUT_PATH)

    if target.exists():
        print(f"Rollout file already exists locally: {target}")
        return target

    if not source.exists():
        raise FileNotFoundError(
            f"Missing rollout data in Drive: {source}. "
            "Upload the rollout JSONL to MyDrive/RL/Data or update ROLLOUT_FILE_NAME."
        )

    shutil.copy2(source, target)
    print(f"Copied rollout data: {source} -> {target}")
    return target


def summarize_rollouts(path):
    groups = read_jsonl(path)
    output_count = sum(len(group.get("outputs", [])) for group in groups)
    useful_groups = 0
    correct_outputs = 0
    format_valid_outputs = 0
    for group in groups:
        outputs = group.get("outputs", [])
        rewards = [output.get("reward") for output in outputs]
        if len(outputs) >= 2 and len(set(rewards)) > 1:
            useful_groups += 1
        correct_outputs += sum(output.get("is_correct", 0) for output in outputs)
        format_valid_outputs += sum(output.get("is_format_valid", 0) for output in outputs)

    return {
        "groups": len(groups),
        "outputs": output_count,
        "useful_reward_variance_groups": useful_groups,
        "correct_outputs": correct_outputs,
        "format_valid_outputs": format_valid_outputs,
    }

## 5. Restore Checkpoint and Rollout Data

In [ ]:
restore_sft_checkpoint()
copy_rollout_data()

rollout_summary = summarize_rollouts(ROLLOUT_PATH)
display(pd.DataFrame([rollout_summary]))

sample_groups = read_jsonl(ROLLOUT_PATH, limit=2)
for group in sample_groups:
    print("=" * 80)
    print("id:", group.get("id"), "ground_truth:", group.get("ground_truth"))
    print("question:", group.get("question"))
    print("num outputs:", len(group.get("outputs", [])))
    print("rewards:", [output.get("reward") for output in group.get("outputs", [])])

## 6. Run DAPO

In [ ]:
if RUN_DAPO_TRAINING:
    command = [
        "python", "-B", "scripts/train_dapo.py",
        "--model-name", MODEL_NAME,
        "--adapter-path", LOCAL_SFT_ADAPTER_PATH,
        "--reference-adapter-path", LOCAL_SFT_ADAPTER_PATH,
        "--rollout-path", ROLLOUT_PATH,
        "--output-dir", DAPO_OUTPUT_DIR,
        "--max-groups", str(MAX_GROUPS),
        "--batch-size", str(BATCH_SIZE),
        "--epochs", str(EPOCHS),
        "--learning-rate", str(LEARNING_RATE),
        "--clip-low", str(CLIP_LOW),
        "--clip-high", str(CLIP_HIGH),
        "--max-length", str(MAX_LENGTH),
    ]
    if not USE_REFERENCE_MODEL:
        command.append("--no-reference-model")

    run_command(command)
else:
    print("RUN_DAPO_TRAINING=False; skipping DAPO training.")

## 7. Inspect Metrics

In [ ]:
metrics_path = Path(DAPO_OUTPUT_DIR) / "dapo_metrics.json"
if metrics_path.exists():
    with metrics_path.open("r", encoding="utf-8") as f:
        metrics = json.load(f)
    display(pd.DataFrame([metrics]))
else:
    print(f"Metrics file not found yet: {metrics_path}")

## 8. Save DAPO Checkpoint to Drive

In [ ]:
if SAVE_DAPO_TO_DRIVE:
    mount_drive_if_needed()
    if not Path(DAPO_OUTPUT_DIR).exists():
        raise FileNotFoundError(f"DAPO output not found: {DAPO_OUTPUT_DIR}")

    DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    archive_base = Path("/tmp") / f"{DAPO_RUN_NAME}_{timestamp}"
    archive_path = shutil.make_archive(str(archive_base), "zip", DAPO_OUTPUT_DIR)
    drive_archive_path = DRIVE_CHECKPOINT_DIR / Path(archive_path).name
    shutil.copy2(archive_path, drive_archive_path)

    print("Saved DAPO checkpoint archive to:")
    print(drive_archive_path)
    print("Archive size MB:", drive_archive_path.stat().st_size / (1024 * 1024))
else:
    print("SAVE_DAPO_TO_DRIVE=False; skipping Drive archive.")